In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -R "cifar_tuning"

In [ ]:
# Copy Keras Tuner Values
!cp -r "/content/drive/My Drive/Colab Notebooks/cifar_tuning/cifar_tuning" "./"

In [ ]:
# Copy cleaned data
!cp "/content/drive/My Drive/Colab Notebooks/cifar10_cleaned.npz" "./cifar10_cleaned.npz"

In [ ]:
!pip install keras_tuner

In [ ]:
# Imports
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
import numpy as np
import os
import pandas as pd
from sklearn.metrics import confusion_matrix
import seaborn as sns


In [ ]:
# Global Parameters
BATCH_SIZE = 128 # Using Google Colab Notebook with T4 GPU, so it is more than capable of using this size.
EPOCHS = 200 # Max Epochs, but early stopping is configured for actual training
NUM_CLASSES = 10 # Number of classes in CIFAR Dataset
INPUT_SHAPE = (32, 32, 3)
DATA_PATH = "./CIFAR-10-C" # Location of CIFAR-10-C
CLASS_NAMES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']


In [ ]:
# Function to load prepared data from data cleaning pipeline
def load_prepared_data(filepath='cifar10_cleaned.npz'):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File {filepath} not found.")

    with np.load(filepath) as data:
        x_train = data['x_train']
        y_train = data['y_train']
        x_test = data['x_test']
        y_test = data['y_test']
        x_val = data['x_val']
        y_val = data['y_val']
    print("Data loaded successfully.")
    print("x_train shape:", x_train.shape)
    print("y_train shape:", y_train.shape)
    print("x_test shape:", x_test.shape)
    print("y_test shape:", y_test.shape)
    print("x_val shape:", x_val.shape)
    print("y_val shape:", y_val.shape)
    return x_train, y_train, x_test, y_test, x_val, y_val

In [ ]:
# Import prepared data from cleaning algorithm
x_train, y_train, x_test, y_test, x_val, y_val = load_prepared_data()

In [ ]:
# Function Defines model. Use_robust allows the heavy augmentation layers to be removed to
# establish baseline and robust models separately.
def build_model(hp, use_robust = True):
    model = models.Sequential()
    model.add(layers.Input(shape=INPUT_SHAPE))

    if use_robust:
        model.add(layers.RandomContrast(0.8))
        model.add(layers.RandomBrightness(0.05))
        model.add(layers.RandomRotation(0.05))
        model.add(layers.RandomZoom(0.1))
        model.add(layers.RandomColorDegeneration(0.02))
        model.add(layers.GaussianNoise(0.05))

    model.add(layers.RandomFlip(mode='horizontal'))

    model.add(layers.RandomTranslation(0.1, 0.1))



    # Block 1 (Edges)
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Hyperparameter: Dropout
    dropout_rate = hp.Float("dropout", 0.0, 0.5, step=0.1)
    model.add(layers.Dropout(dropout_rate))

    # Block 2 (Shapes)
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(dropout_rate))

    # Block 3 (Objects)
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(dropout_rate))

    # Head
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # Hyperparameter: Learning Rate
    lr = hp.Float("learning_rate", 0.00001, 0.001, sampling="log")

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=['accuracy'])

    return model

In [ ]:
# Load best hyper parameters for robust model

tuner_robust = kt.BayesianOptimization(
    hypermodel=lambda hp: build_model(hp, use_robust = True),
    objective='val_accuracy',
    max_trials=24,
    directory='cifar_tuning',
    project_name='baysean_opt_robust_v2.0',
    num_initial_points=6,
    overwrite=False, # This allows the search to be skipped if it has already been run
    alpha=0.001,
    beta=5.0,
    seed=42
)

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Perform Bayesean Optimization of parameters to find best LR and dropout parameters on Robust model
tuner_robust.search(
    x_train, y_train,
    epochs=30,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=2
)

best_hps_robust = tuner_robust.get_best_hyperparameters()[0]
print(f"Best Robust Learning Rate: {best_hps_robust.get('learning_rate')}")
print(f"Best Robust Dropout Rate: {best_hps_robust.get('dropout')}")

In [ ]:
# Configure callbacks to include dynamic reduction of LR on stagnation and early stopping.
callbacks_list = [
    callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=7, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1)
]

In [ ]:
# Train Robust Model
print("STARTING ROBUST TRAINING...")
model_robust = build_model(best_hps_robust, use_robust=True)
history_robust = model_robust.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=callbacks_list,
    verbose=2
)
model_robust.save('final_robust.keras')

In [ ]:
# Set up the plot style
plt.style.use('ggplot')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# A. Loss Curves
ax1.plot(history_robust.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history_robust.history['val_loss'], label='Validation Loss', linestyle='--', linewidth=2)
ax1.set_title('Loss Curves (Categorical Crossentropy)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# B. Accuracy Curves
ax2.plot(history_robust.history['accuracy'], label='Training Accuracy', linewidth=2)
ax2.plot(history_robust.history['val_accuracy'], label='Validation Accuracy', linestyle='--', linewidth=2)
ax2.set_title('Accuracy Curves')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# C. Confusion Matrix
print("\nGenerating Confusion Matrix...")
# Get predictions
y_pred_probs = model_robust.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.argmax(axis=1)

# Calculate Matrix
cm = confusion_matrix(y_true, y_pred)

# Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: Final Robust Model')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()